# NIOS PDF Extraction — marker-pdf on Kaggle

**What this notebook does:**
1. Reads chapter PDF URLs from `chapter_urls/<subject>.json` (uploaded as a Kaggle dataset input)
2. Downloads each chapter PDF directly from the NIOS website
3. Runs `marker-pdf` on each PDF to produce structured JSON (typed blocks: Section-header, Text, Equation, Table, etc.)
4. Saves one `Chapter N.json` per chapter to `/kaggle/working/extracted/`

**Setup:**
- Add the `nios-<subject>-urls` dataset as an input (contains the `<subject>.json` URL config)
- Enable GPU accelerator (T4) for faster extraction
- Enable Internet access

**After this runs:**  
The output files become a Kaggle dataset. Locally run:
```bash
python 02_extract/download_from_kaggle.py --subject maths-12 --dataset <username>/nios-maths-12-extracted
```

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set the subject ID to match the JSON file in your input dataset
SUBJECT = "maths-12"

# Path to the chapter URLs JSON uploaded as a Kaggle dataset input
# The dataset should be named: nios-<subject>-urls
URLS_JSON = f"/kaggle/input/nios-{SUBJECT}-urls/{SUBJECT}.json"

# Output directory (becomes part of the Kaggle output dataset)
OUTPUT_DIR = f"/kaggle/working/extracted/{SUBJECT}"

# Temp directory for downloaded PDFs (not saved as output)
PDF_DIR = f"/kaggle/working/pdfs/{SUBJECT}"

# marker output format
MARKER_FORMAT = "json"  # options: json, markdown, html

# Resume: skip chapters already extracted
RESUME = True

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# marker-pdf includes torch, transformers etc. — takes ~3 min
!pip install -q marker-pdf requests

In [ ]:
import json
import os
import shutil
import sys
import time
from pathlib import Path

import requests

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)

# ── Load chapter URL config ───────────────────────────────────────────────────
with open(URLS_JSON) as f:
    config = json.load(f)

chapters = config["chapters"]
subject_name = config["subject_name"]
class_level = config["class_level"]

print(f"Subject:  {subject_name} — Class {class_level}")
print(f"Chapters: {len(chapters)}")
print()
for ch in chapters:
    print(f"  {ch['name']}")

In [ ]:
# ── Download PDFs from NIOS ───────────────────────────────────────────────────
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

def download_pdf(url: str, dest: Path, max_retries: int = 4) -> bool:
    """Download a PDF from URL to dest. Returns True on success."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
            resp.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        except Exception as e:
            wait = 2 ** attempt
            print(f"    ⚠️  Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return False


download_stats = {"ok": 0, "skip": 0, "fail": 0}

for ch in chapters:
    pdf_path = Path(PDF_DIR) / ch["name"]

    if RESUME and pdf_path.exists() and pdf_path.stat().st_size > 1024:
        print(f"  ⏭️  {ch['name']} (already downloaded)")
        download_stats["skip"] += 1
        continue

    print(f"  📥 Downloading {ch['name']} ...")
    ok = download_pdf(ch["url"], pdf_path)
    if ok:
        size_kb = pdf_path.stat().st_size / 1024
        print(f"     ✅ {size_kb:.0f} KB")
        download_stats["ok"] += 1
    else:
        print(f"     ❌ Failed to download {ch['name']}")
        download_stats["fail"] += 1

print(f"\nDownload summary: ✅ {download_stats['ok']}  ⏭️ {download_stats['skip']}  ❌ {download_stats['fail']}")

In [ ]:
# ── Extract with marker-pdf ───────────────────────────────────────────────────
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

# Load models once (expensive — ~1-2 min)
print("🔧 Loading marker models...")
config_parser = ConfigParser({"output_format": MARKER_FORMAT})
models = create_model_dict()
print("✅ Models loaded\n")


def extract_chapter(pdf_path: Path, out_dir: Path) -> bool:
    """
    Run marker on a single PDF and save the JSON output.
    Returns True on success.
    """
    stem = pdf_path.stem  # e.g. "Chapter 1"
    out_file = out_dir / f"{stem}.json"

    if RESUME and out_file.exists() and out_file.stat().st_size > 100:
        print(f"  ⏭️  {stem} (already extracted)")
        return True

    print(f"  ⚙️  Extracting {stem} ...")
    try:
        converter = PdfConverter(
            config=config_parser.generate_config_dict(),
            artifact_dict=models,
            processor_list=config_parser.get_processors(),
            renderer=config_parser.get_renderer(),
        )
        rendered = converter(str(pdf_path))
        text, _, images = text_from_rendered(rendered)

        # text is the JSON string when output_format=json
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(text)

        size_kb = out_file.stat().st_size / 1024
        print(f"     ✅ {size_kb:.0f} KB → {out_file.name}")
        return True
    except Exception as e:
        print(f"     ❌ Failed: {e}")
        return False


extract_stats = {"ok": 0, "skip": 0, "fail": 0, "failed_chapters": []}
out_dir = Path(OUTPUT_DIR)

# Process chapters in order
pdfs = sorted(
    Path(PDF_DIR).glob("*.pdf"),
    key=lambda p: (
        0 if p.stem.startswith("Chapter") else 1,
        int(__import__("re").search(r"\d+", p.stem).group())
        if __import__("re").search(r"\d+", p.stem) else 999,
    )
)

if not pdfs:
    print("❌ No PDFs found. Check the download step above.")
else:
    print(f"📄 Extracting {len(pdfs)} PDFs...\n")
    for pdf in pdfs:
        ok = extract_chapter(pdf, out_dir)
        if ok:
            out_file = out_dir / f"{pdf.stem}.json"
            if out_file.exists():
                extract_stats["ok"] += 1
            else:
                extract_stats["skip"] += 1
        else:
            extract_stats["fail"] += 1
            extract_stats["failed_chapters"].append(pdf.stem)

    print(f"\nExtraction summary: ✅ {extract_stats['ok']}  ⏭️ {extract_stats['skip']}  ❌ {extract_stats['fail']}")
    if extract_stats["failed_chapters"]:
        print(f"Failed chapters: {extract_stats['failed_chapters']}")

In [ ]:
# ── Write manifest ────────────────────────────────────────────────────────────
import datetime

json_files = sorted(out_dir.glob("*.json"))
manifest = {
    "subject": SUBJECT,
    "subject_name": subject_name,
    "class_level": class_level,
    "extracted_at": datetime.datetime.utcnow().isoformat() + "Z",
    "marker_format": MARKER_FORMAT,
    "chapters": [
        {"name": f.stem, "file": f.name, "size_bytes": f.stat().st_size}
        for f in json_files
        if f.name != "_manifest.json"
    ],
}

with open(out_dir / "_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"📋 Manifest written: {len(manifest['chapters'])} chapter(s)")
print(f"📂 Output dir: {OUTPUT_DIR}")
print()
for ch in manifest["chapters"]:
    size_kb = ch["size_bytes"] / 1024
    print(f"  ✅ {ch['name']}.json ({size_kb:.0f} KB)")

In [ ]:
# ── Preview one chapter block structure ──────────────────────────────────────
import json, re
from pathlib import Path

# Find the first chapter JSON and show structure
chapter_files = sorted(
    [f for f in Path(OUTPUT_DIR).glob("Chapter *.json")],
    key=lambda p: int(re.search(r"\d+", p.stem).group())
)

if chapter_files:
    sample = chapter_files[0]
    with open(sample) as f:
        data = json.load(f)

    print(f"Preview: {sample.name}")
    print(f"Top-level keys: {list(data.keys())}")
    print()

    # Show first few blocks to verify structure
    blocks = data.get("blocks", [])
    print(f"Total blocks: {len(blocks)}")
    print("\nFirst 5 blocks:")
    for b in blocks[:5]:
        btype = b.get("block_type", b.get("type", "unknown"))
        raw_lines = b.get("text_lines", []) or b.get("lines", [])
        text_preview = " ".join(
            line.get("text", "") if isinstance(line, dict) else str(line)
            for line in (raw_lines[:2] if raw_lines else [])
        )[:120]
        print(f"  [{btype}] {text_preview}")